# Exercise 12 – Feature engineering 

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

In [2]:
import numpy as np                  # Equivalent to LinearAlgebra / Statistics
import matplotlib.pyplot as plt     # Equivalent to Plots
import h5py                         # Equivalent to HDF5
import scipy.io as sio              # Equivalent to MAT (for loading .mat files)
import torch                        # Equivalent to Flux (PyTorch is the closest DL analog)
import pickle                       # Equivalent to Serialization
import random                       # Equivalent to Random (or you can use np.random)
import statistics                   # Equivalent to Statistics (though numpy is often used)
from sklearn.model_selection import train_test_split # Example of MLUtils equivalent
# from torch.utils.data import DataLoader, Dataset   # Another MLUtils equivalent depending on use case

## Task 1 – Extended Dynamic Mode Decomposition using a dictionary
Learn a linear dynamical system from trajectory data via the Dynamic Mode Decomposition. As the example, we will use the Duffing oscillator with a two-dimensional state $x\in\mathbb{R}^2$. The function for the right-hand side is given below

### Understanding Extended Dynamic Mode Decomposition (EDMD)

Extended Dynamic Mode Decomposition (EDMD) is a data-driven algorithm used to discover the underlying dynamics of a system from data. It is particularly powerful for **nonlinear systems**, like the Duffing oscillator in your example.

A breakdown of how it works and why the "dictionary" is the core of the method:

* **The Limitation of Standard DMD:** Standard Dynamic Mode Decomposition (DMD) tries to find a best-fit linear matrix $A$ that maps the current state to the next state ($X_{k+1} \approx A X_k$). If the system is highly nonlinear, a simple linear fit in the original state space (e.g., $x \in \mathbb{R}^2$) will fail to capture the true dynamics accurately.
* **The Koopman Perspective:** EDMD is built on Koopman operator theory. The central idea is that a finite-dimensional *nonlinear* system can be perfectly represented as an infinite-dimensional *linear* system by looking at "observables" (functions of the state) rather than the state itself.
* **The "Dictionary" (Lifting):** Because we cannot compute infinite dimensions, we choose a finite set of observable functions to map (or "lift") our original state variables into a higher-dimensional feature space. **This set of chosen functions is called the dictionary** (often denoted as $\Psi(x)$).
* *Example:* For a 2D state $x = [x_1, x_2]$, a polynomial dictionary of up to degree 2 would be: $\Psi(x) = [1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$.


* **The Linear Fit in Lifted Space:** Once you push all your snapshot data through this dictionary, EDMD performs linear regression (usually via least squares) on these new, high-dimensional lifted states. It finds a matrix $K$ that steps the observables forward in time: $\Psi(X_{k+1}) \approx K \Psi(X_k)$.

By doing the math in this lifted dictionary space, EDMD can capture complex nonlinear behaviors using linear algebraic techniques.

In [3]:
def rhs(x):
    """
    Vectorized right-hand side of the Duffing oscillator.
    Accepts an array x of shape (2, N).
    """
    a, beta, delta = -1, 1, 0.1
    
    x1 = x[0, :]
    x2 = x[1, :]
    
    xdot1 = x2
    xdot2 = -delta * x2 - a * x1 - beta * x1**3
    
    # vstack stacks the 1D arrays back into a (2, N) array
    return np.vstack((xdot1, xdot2))

The "Right-Hand Side" (RHS) of the Duffing oscillator refers to the mathematical expression of the system's dynamics when formulated as a set of first-order ordinary differential equations (ODEs) in state-space form.

The classic, unforced Duffing oscillator is described by the following second-order non-linear differential equation:

$$\ddot{x} + \delta \dot{x} + \alpha x + \beta x^3 = 0$$

To find the RHS, we define the state variables. Let $x_1 = x$ (position) and $x_2 = \dot{x}$ (velocity or momentum). We can then rewrite the second-order ODE as a system of two first-order ODEs:

$$\dot{x}_1 = x_2$$

$$\dot{x}_2 = -\delta x_2 - \alpha x_1 - \beta x_1^3$$

The **RHS** is the vector-valued function $f(\mathbf{x})$ that computes the time derivative of the state vector $\mathbf{x} = [x_1, x_2]^T$:

$$f(\mathbf{x}) = \begin{bmatrix} x_2 \\ -\delta x_2 - \alpha x_1 - \beta x_1^3 \end{bmatrix}$$

### What the parameters mean:

* **$\alpha$ (Linear stiffness):** Controls the standard linear restoring force, analogous to a standard Hooke's law spring.
* **$\beta$ (Non-linear stiffness):** Controls the non-linear cubic restoring force. If $\beta > 0$, the "spring" stiffens as it stretches (hardening spring).
* **$\delta$ (Damping):** The coefficient of friction or energy dissipation.

In the specific Julia code you shared earlier, the parameters were set to $\alpha = -1$, $\beta = 1$, and $\delta = 0.1$. When $\alpha$ is negative and $\beta$ is positive, the system models a mass moving in a "double-well potential"—meaning there are two stable equilibrium points that the system can settle into, which is a common benchmark for evaluating non-linear system identification methods.

a) Create training data $X$ and $X’$. To this end, draw $N=1000$ initial conditions randomly and uniformly from the rectangle $[-2,2] \times [-2,2]$ and perform one time step using the explicit Euler scheme with a time step of $h = 0.1$. 

In [4]:
# --- Parameters ---
N = 1000
h = 0.1

# --- 1. Create training data X ---
# We use shape (2, N) so that each column represents one state snapshot [x1, x2].
# This column-wise structure is standard for DMD data matrices.
np.random.seed(42) # Seed for reproducibility
X = np.random.uniform(low=-2.0, high=2.0, size=(2, N))

# --- 2. Create X' using explicit Euler ---
# The explicit Euler scheme: x_{k+1} = x_k + h * f(x_k)
X_prime = X + h * rhs(X)

# Verify the shapes
print(f"Shape of X: {X.shape}")
print(f"Shape of X': {X_prime.shape}")

Shape of X: (2, 1000)
Shape of X': (2, 1000)


In [5]:
# Asserting the shapes
assert X.shape == (2, 1000), f"Expected X shape (2, 1000), got {X.shape}"
assert X_prime.shape == (2, 1000), f"Expected X_prime shape (2, 1000), got {X_prime.shape}"

### BEGIN TESTS

# Check if sample points are in range
# np.all() checks if every element in the boolean array is True
assert np.all((X >= -2) & (X <= 2)), "Not all sample points are within the range [-2, 2]"

# Check if X_prime is calculated correctly
# np.testing.assert_allclose is the Python equivalent of Julia's isapprox for arrays.
# It checks if two arrays are equal within a small tolerance.
np.testing.assert_allclose(
    X_prime, 
    X + 0.1 * rhs(X), 
    err_msg="X_prime was not calculated correctly using the explicit Euler scheme"
)

### END TESTS

b) Use DMD in its standard form to learn a linear system with $A\in\mathbb{R}^{2\times 2}$. Simulate the dynamics over $m=100$ time steps using the initial condition $x_0=[1.0, -1.0]$. Report on the RMSE between the created trajectory and a true trajectory of the system (same initial condition, explicit Euler integration).

Note: To avoid confusion regarding the RMSE on multiple dimensions, take this one:
$ \text{RMSE} = \sqrt{ \frac{1}{T} \sum_{t = 1}^{T} \sum_{i = 1}^{d} (X_{t, i} - Y_{t, i})^2 } $

### Understanding the Concept

**Standard Dynamic Mode Decomposition (DMD)**
The goal of standard DMD here is to find a linear representation of your non-linear system. Given your snapshot matrices `X` and `X_prime`, you are looking for a system matrix, denoted here as $\hat{\hat{A}}$, that maps the current state to the next state:


$$X' \approx \hat{\hat{A}} X$$


To find the best-fit $\hat{\hat{A}}$ in a least-squares sense, we use the Moore-Penrose pseudoinverse (denoted as $\dagger$):


$$\hat{\hat{A}} = X' X^\dagger$$

**Simulation & Comparison**
Once you have learned $\hat{\hat{A}}$, you can predict future states from an initial condition `x0` strictly through repeated matrix multiplication (where the next state is $\hat{\hat{A}}$ times the current state).
You are asked to compare this linearized, data-driven trajectory against the "true" trajectory, which is generated by feeding the exact same initial condition through your explicit Euler scheme using the non-linear right-hand side equations from your earlier steps.

**RMSE (Root Mean Square Error)**
The provided formula calculates the squared Euclidean distance between the true and predicted state at each time step, averages those squared distances over the entire simulation window of `m=100` steps, and takes the square root. It is a standard metric to quantify how quickly the linear DMD approximation diverges from the true non-linear dynamics.

In [6]:
# Initial setup
x0 = np.array([1.0, -1.0])
m = 100

### BEGIN SOLUTION

# 1. Learn the linear system matrix A using the pseudoinverse
A = X_prime @ np.linalg.pinv(X)

# 2. Initialize arrays to store the trajectories
# Shape is (2, 100) to hold 2 dimensions over m time steps
dmd_trajectory = np.zeros((2, m))
euler_trajectory = np.zeros((2, m))

# Set the starting points
x_dmd = x0.copy()
x_euler = x0.copy()

# 3. Simulate the dynamics over m time steps
for t in range(m):
    # DMD step: purely linear matrix multiplication
    x_dmd = A @ x_dmd
    
    # Euler step: using the non-linear RHS
    # Reshaping x_euler to (2, 1) ensures it works with our previously vectorized rhs function
    x_euler_col = x_euler.reshape(2, 1)
    x_euler_next = x_euler_col + h * rhs(x_euler_col)
    x_euler = x_euler_next.flatten()
    
    # Store the results column by column
    dmd_trajectory[:, t] = x_dmd
    euler_trajectory[:, t] = x_euler

# 4. Calculate the RMSE
# First, calculate the squared differences
squared_errors = (dmd_trajectory - euler_trajectory)**2

# Sum across the dimensions (which is axis 0 for our 2x100 arrays)
sum_over_dimensions = np.sum(squared_errors, axis=0)

# Mean across the time steps, then square root
RMSE = np.sqrt(np.mean(sum_over_dimensions))

### END SOLUTION

print(f"Learned Matrix A:\n{A}")
print(f"RMSE: {RMSE:.4f}")

Learned Matrix A:
[[ 1.          0.1       ]
 [-0.14138785  0.9856344 ]]
RMSE: 2.1346


c) Implement a dictionary of radial basis functions (RBFs), i.e.,  
$$ 
\varphi_c(x) = \exp(- \gamma \| x - c \|_2^2 ),
$$ 
$$ 
z = \Psi(x) = [\varphi_{c_1}(x), \ldots, \varphi_{c_r}(x)],
$$ 
where the $c_i\in\mathbb{R}^2$ are the centers of the individual RBFs and $\gamma>0$ is a hyperparameter determining the width of the RBF 

### Explaining the Concepts

**1. The Goal: Lifting the State Space**
In standard DMD, we attempt to find linear relationships directly in the original state space. For highly nonlinear systems, this often fails. Extended DMD (EDMD) solves this by mapping (or "lifting") the original low-dimensional state $x$ into a much higher-dimensional feature space using a set of non-linear functions. This collection of functions is the **dictionary**, denoted as $\Psi(x)$.

**2. Radial Basis Functions (RBFs)**
A Radial Basis Function is a specific type of function whose output depends solely on the distance between the input $x$ and a fixed origin point (the "center" $c$).

The specific RBF used here is the Gaussian function:


$$\varphi_c(x) = \exp(-\gamma ||x - c||_2^2)$$

* **$x$:** The current state vector (e.g., a specific point in a 2D space).
* **$c$:** The center of the RBF. You define multiple centers scattered across your state space.
* **$||x - c||_2^2$:** The squared Euclidean distance between the state and the center.
* **$\gamma$ (gamma):** A hyperparameter that determines the "width" of the basis function. A smaller $\gamma$ means the function is wide and influences a large area; a larger $\gamma$ makes it narrow and highly localized.

**3. The Lifted State ($z$)**
Instead of representing the system's state just by its coordinates $x \in \mathbb{R}^2$, we represent it by how it activates a grid of RBFs. If you have $r$ centers ($c_1$ through $c_r$), your new lifted state $z$ is a vector of length $r$:


$$z = \Psi(x) = [\varphi_{c_1}(x), \dots, \varphi_{c_r}(x)]$$

When the state $x$ is very close to a specific center $c_i$, that particular RBF will output a value close to 1. As $x$ moves away, the output decays toward 0. EDMD will then perform linear regression on these $z$ vectors rather than the raw $x$ vectors.

In [7]:
# Function to compute the lifted state using a dictionary of RBFs
def rbf_dictionary(x, centers, gamma=1.5):
    ### BEGIN SOLUTION
    # Convert lists to numpy arrays for vectorized operations
    x_arr = np.array(x)
    c_arr = np.array(centers)
    
    # Calculate the squared Euclidean distance: ||x - c||^2
    # By subtracting x_arr from c_arr, numpy broadcasts x_arr to match the number of centers.
    # We then square the differences and sum along axis 1 (the coordinates axis)
    distances_sq = np.sum((c_arr - x_arr)**2, axis=1)
    
    # Compute the RBF values
    z = np.exp(-gamma * distances_sq)
    
    # Convert back to a standard Python list if strict matching to the expected format is needed, 
    # though keeping it as a numpy array is usually preferred in ML/data science pipelines.
    return z.tolist() 
    ### END SOLUTION

In [8]:
# --- BEGIN TESTS ---
# 1. Check if it's a callable function (Python equivalent of checking if it's a Function type)
assert callable(rbf_dictionary)

# 2. Check the length of the returned array matches the number of centers
assert len(rbf_dictionary([1, 1], [[2, 2], [3, 3]])) == 2
assert len(rbf_dictionary([1, 1], [[2, 2], [3, 3], [4, 4]])) == 3

# 3. Check numerical approximation (Python equivalent of isapprox)
np.testing.assert_allclose(
    rbf_dictionary([1, 1], [[2, 2], [3, 3]]), 
    [0.049787068367863896, 6.144212353328188e-6]
)

np.testing.assert_allclose(
    rbf_dictionary([0, 0], [[2, 2], [2, -2], [-2, 2], [-2, -2]]), 
    [6.144212353328188e-6, 6.144212353328188e-6, 6.144212353328188e-6, 6.144212353328188e-6]
)

print("All tests passed successfully!")
# --- END TESTS

All tests passed successfully!


d) Use the extended version of DMD (aka EDMD), where you find a linear system for the lifted state $\Psi(x)=z\in\mathbb{R}^r$ instead of $x\in\mathbb{R}^2$ (use the trajectory from a) ). To this end, introduce $r=100$ centers on an equidistant $10 \times 10$ grid in the area $[-2, 2] \times [-2, 2]$, with $\gamma = 1$. Train the corresponding matrix $A\in\mathbb{R}^{100 \times 100}$. 

### Explaining the Problem

In the previous steps, you used standard DMD to find a linear mapping directly in the 2D state space ($X' \approx \hat{\hat{A}} X$). Because the Duffing oscillator is highly nonlinear, a $2 \times 2$ matrix will struggle to capture its true dynamics accurately over time.

Extended Dynamic Mode Decomposition (EDMD) solves this by mapping the problem into a higher-dimensional space where the dynamics behave more linearly.

Here is the step-by-step breakdown of what task (d) is asking:

1. **Define the Lifted Space:** Instead of working with the 2D coordinates $x$, you are moving to a 100-dimensional space. The dimensions of this new space are defined by 100 Radial Basis Functions (RBFs).
2. **Create the Grid (Centers):** To distribute these RBFs evenly, you need to create a $10 \times 10$ equidistant grid over your state space $[-2, 2] \times [-2, 2]$. Every intersection on this grid becomes a "center" ($c$) for one of your 100 RBFs.
3. **Lift the Data ($Z$ and $Z'$):** You will pass every $2 \times 1$ snapshot in your original $X$ matrix through your dictionary function to get a $100 \times 1$ vector. Doing this for all 1000 snapshots gives you the lifted data matrix $Z \in \mathbb{R}^{100 \times 1000}$. You do the exact same for $X'$ to get $Z'$.
4. **Train the Model:** Finally, you find the $100 \times 100$ matrix $\hat{\hat{A}}_{edmd}$ that best maps the current lifted state to the next lifted state ($Z' \approx \hat{\hat{A}}_{edmd} Z$). This is solved exactly like standard DMD using the Moore-Penrose pseudoinverse.

In [10]:
### BEGIN SOLUTION

# 1. Create the 10x10 equidistant grid of centers
# np.linspace creates 10 evenly spaced points between -2 and 2
grid_1d = np.linspace(-2, 2, 10)

# np.meshgrid creates full 2D coordinate matrices
X1, X2 = np.meshgrid(grid_1d, grid_1d)

# Flatten and stack them to get a list of 100 coordinates, shape: (100, 2)
centers = np.column_stack((X1.ravel(), X2.ravel()))

# 2. Compute the lifted states Z and Z'
# We iterate over the columns (snapshots) of X and X_prime.
# The resulting lists are transposed (.T) so that each lifted state is a column,
# resulting in matrices of shape (100, 1000).
Z = np.array([
    rbf_dictionary(X[:, i], centers, gamma=1.0) 
    for i in range(X.shape[1])
]).T

Z_prime = np.array([
    rbf_dictionary(X_prime[:, i], centers, gamma=1.0) 
    for i in range(X_prime.shape[1])
]).T

# 3. Train the EDMD matrix A_edmd using the pseudoinverse
A_edmd = Z_prime @ np.linalg.pinv(Z)

### END SOLUTION

# Optional: Verify the shapes
print(f"Shape of centers: {centers.shape}")
print(f"Shape of Z: {Z.shape}")
print(f"Shape of Z_prime: {Z_prime.shape}")
print(f"Shape of A_edmd: {A_edmd.shape}")

Shape of centers: (100, 2)
Shape of Z: (100, 1000)
Shape of Z_prime: (100, 1000)
Shape of A_edmd: (100, 100)


e) Implement a projection operation that maps from $z$ to $x$, such that you can reconstruct the original state $x$ from predictions of the lifted state $z$. This step can be realized by a linear mapping $x = P z$, where the projection matrix $P\in\mathbb{R}^{2\times r}$ can be determined using linear regression. 

### Explaining the Problem

In the previous step, you successfully mapped (or "lifted") your 2D state $x$ into a 100-dimensional RBF feature space $z$. You then learned a matrix $A_{edmd}$ that predicts future states *in that high-dimensional space* ($Z' \approx A_{edmd} Z$).

However, predicting how 100 Radial Basis Functions will activate in the future isn't practically useful on its own. To visualize the trajectory of the Duffing oscillator or calculate the error, you need to bring that high-dimensional prediction back down to the original 2D physical state (position and velocity).

You need a **projection matrix**, $P$.

Because you have the original training data $X \in \mathbb{R}^{2 \times N}$ and you have already calculated the corresponding lifted data $Z \in \mathbb{R}^{r \times N}$ (where $r=100$ and $N=1000$), you can find the relationship between them using linear regression:


$$X \approx P Z$$

To isolate $P$ and find the best-fit matrix that maps $Z$ back to $X$, you multiply both sides by the Moore-Penrose pseudoinverse of $Z$ (denoted as $Z^\dagger$):


$$P = X Z^\dagger$$

Once you have $P$, anytime your EDMD model predicts a future state $z_{k+1}$ in the 100-dimensional space, you can simply multiply it by $P$ to recover the 2D prediction: $x_{k+1} = P z_{k+1}$.

In [11]:
### BEGIN SOLUTION

# Determine the projection matrix P using linear regression via the pseudoinverse.
# X has shape (2, 1000), Z has shape (100, 1000).
# np.linalg.pinv(Z) will have shape (1000, 100).
# The resulting P will correctly have shape (2, 100).

P = X @ np.linalg.pinv(Z)

### END SOLUTION

# Verify the shape
print(f"Shape of projection matrix P: {P.shape}")

Shape of projection matrix P: (2, 100)


f) Repeat the experiment from b), but now compare the original trajectory to the one obtained using EDMD and the consecutive projection step to recover $x$. 

```Hint: Don't forget to transform the initial condition accordingly```

### Explaining the Problem

This task brings everything together. You are going to test how well your Extended Dynamic Mode Decomposition (EDMD) model predicts the system's behavior compared to the "true" trajectory, just like you did with standard DMD in task (b).

Here is the step-by-step logic:

1. **Lift the Initial Condition:** Your simulation starts at $x_0 = [1.0, -1.0]$. Because your EDMD matrix ($A_{edmd}$) operates in the 100-dimensional RBF space, you cannot multiply it directly by $x_0$. You must first pass $x_0$ through your dictionary to get the lifted starting point, $z_0$.
2. **Simulate the Lifted Trajectory:** Using $z_0$, you simulate $m=100$ time steps purely through linear matrix multiplication in the high-dimensional space: $z_{k+1} = A_{edmd} z_k$. This builds your `edmd_trajectory_lifted`.
3. **Project Back to 2D:** The 100-dimensional predictions need to be translated back into physical coordinates (position and velocity) so you can compare them to the actual Duffing oscillator. You multiply the entire lifted trajectory by the projection matrix $P$ you found in step (e) to get `edmd_trajectory_projected`.
4. **Calculate the Error:** Finally, you calculate the Root Mean Square Error (RMSE) between your projected 2D EDMD trajectory and the true `euler_trajectory` calculated back in step (b).

If EDMD worked correctly, this `error_edmd` should be significantly lower than the RMSE you got from standard DMD.

In [12]:
### BEGIN SOLUTION

# 1. Initialize arrays to store the trajectories
# Lifted space has 100 dimensions, physical space has 2
edmd_trajectory_lifted = np.zeros((100, m))
edmd_trajectory_projected = np.zeros((2, m))

# 2. Transform the initial condition to the lifted space
# Note: gamma should match what you used during training (e.g., 1.0)
z0 = np.array(rbf_dictionary(x0, centers, gamma=1.0))
z_current = z0.copy()

# 3. Simulate the dynamics in the lifted space over m time steps
for t in range(m):
    # Step forward in time using the learned EDMD matrix
    z_current = A_edmd @ z_current
    
    # Store the 100-dimensional state column by column
    edmd_trajectory_lifted[:, t] = z_current

# 4. Project the entire lifted trajectory back to the original 2D state space
# Shape math: (2, 100) @ (100, 100) = (2, 100)
edmd_trajectory_projected = P @ edmd_trajectory_lifted

# 5. Calculate the RMSE against the true Euler trajectory from part b
squared_errors_edmd = (edmd_trajectory_projected - euler_trajectory)**2
sum_over_dimensions_edmd = np.sum(squared_errors_edmd, axis=0)
error_edmd = np.sqrt(np.mean(sum_over_dimensions_edmd))

### END SOLUTION

print(f"EDMD RMSE: {error_edmd:.6f}")

EDMD RMSE: 1.187120


## Task 2 – Comparing SVD and Autoencoders
In this exercise, we want to compare the capabilities of the Singular Value Decomposition (SVD) and an autoencoder in terms of data compression of high-dimensional trajectory data. The following data set $X\in\mathbb{R}^{101 \times 6001}$ has been created by simulating the Burgers PDE (https://en.wikipedia.org/wiki/Burgers%27_equation) on an $n = 101$-dimensional spatial grid for $N=6000$ time steps. 

a) Use the SVD to compress the $n$-dimensional state into a much lower dimension $k\ll n$. Select the smallest value for $k$ for which the reconstruction error between the original data $x$ and the reconstructed data $\tilde{x}$ is less than 0.1%, i.e. 
$$ 
\frac{ \sum_{i=1}^N \|x_i – \tilde{x}_i \|_2^2 }{ \sum_{i=1}^N \|x_i \|_2^2 } < 0.001. 
$$ 

In [ ]:
#Data loading
file = matopen("burgers.mat")
X = read(file, "u")
close(file)

In [ ]:
function loss_function(original, reconstructed)
    ### BEGIN SOLUTION
   
    ### END SOLUTION
end

In [ ]:
@assert isa(loss_function, Function)

### BEGIN TESTS
@assert isapprox(0.0035087719298245615, loss_function([1 2 3; 4 5 6; 7 8 9], [1 2 3; 4 5 6; 7 8 10]))
### END TESTS

In [ ]:
k = nothing
X_reconstructed_k = nothing

### BEGIN SOLUTION


### END SOLUTION

In [ ]:
#Random SVD sample comparison
random_index = rand(1:size(X,2))
x = range(0.1,10.1, length = 101)
println("Original Image")
Plots.plot(x,X[:, random_index])
println("Reconstructed Image")
display(Plots.plot!(x,X_reconstructed_k[:, random_index]))

b) Now compare this to an autoencoder architecture of your choice. The only constraints that should be respected are: 
- Use only fully connected feed-forward layers (i.e., no convolutions) 
- the bottleneck layer connecting the encoder and the decoder (i.e., the smallest layer whose latent state is the compressed state) has to have dimension at most $r=5$. 
This task is fulfilled if your trained architecture can satisfy the error criterion from a) 

In [ ]:
#define your autoencoder
autoencoder = nothing

### BEGIN SOLUTION

### END SOLUTION


In [ ]:
@assert isa(autoencoder, Flux.Chain)

In [ ]:
# Train your autoencoder.

### BEGIN SOLUTION

### END SOLUTION

In [ ]:
#Random autoencoder sample comparison
x = range(0.1,10.1, length = 101)
random_index = rand(1:size(X,2))
original = X[:,random_index]
reconstruction = autoencoder(original)
Plots.plot(x,original)
display(Plots.plot!(x,reconstruction))